# Tool Patterns Agent — 四种可调用 Tool 模式的演示

本 notebook 演示 LangGraph 中可调用 tool 的 agent 的四种设计模式：

| 模式 | 名称 | 示例工具 | 行为 |
|------|------|----------|------|
| 一 | **函数式** | `search_web`, `fetch_page` | `@tool def fn(args)` — 真实执行外部操作 |
| 二 | **Schema 式** | `DelegateResearch`, `TaskComplete` | `@tool class X(BaseModel)` — 定义 schema，触发子流程/路由 |
| 三 | **伪工具** | `Think`, `QualityCheck` | 不执行外部操作，只记录到 state |
| 四 | **混合模式** | 以上全部 | 三种模式在同一个 tool_executor 中共存 |

代码路径：`deep_research/agents/tool_patterns_agent.py`

In [ ]:
import sys
sys.path.insert(0, '.')

from deep_research.agents.tool_patterns_agent import (
    tool_patterns_agent,
    all_tools,
    tools_by_name,
    ToolPatternsState,
    print_graph,
)
from langchain_core.messages import HumanMessage

print("✓ 导入成功")

## 1. 打印 Agent 的图结构

看 agent 的组织方式：agent（LLM决策）→ tool_executor（分流执行）→ 循环或结束

In [ ]:
print_graph()

## 2. 列出所有注册的工具及其模式分类

In [ ]:
print(f"共注册 {len(all_tools)} 个工具：")
print()
for t in all_tools:
    name = t.name if hasattr(t, 'name') else t.__name__
    doc = (t.__doc__ or '').strip().split('\n')[0][:80]
    print(f"  {name:25s} → {doc}")

## 3. 运行 agent：测试不同场景

### 场景 A：简单问题，只需要搜索

In [ ]:
import nest_asyncio
nest_asyncio.apply()

state_a: ToolPatternsState = {
    "messages": [HumanMessage(content="英伟达最新季度财报的营收数据是多少？")],
    "reflection_history": [],
    "pending_delegations": [],
    "final_answer": "",
    "iteration": 0,
}

result_a = await tool_patterns_agent.ainvoke(state_a)

print(f"迭代次数：{result_a.get('iteration', 0)}")
print(f"思考记录：{len(result_a.get('reflection_history', []))} 条")
print(f"最终答案：{result_a.get('final_answer', '未调用 TaskComplete')[:200]}")
print()
print("--- 思考历史 ---")
for i, ref in enumerate(result_a.get('reflection_history', []), 1):
    print(f"  [{i}] {ref[:150]}...")

### 场景 B：复杂问题，需要委派子任务

In [ ]:
state_b: ToolPatternsState = {
    "messages": [HumanMessage(content="对比分析 OpenAI GPT-4o、Google Gemini、Anthropic Claude 的编程能力差异")],
    "reflection_history": [],
    "pending_delegations": [],
    "final_answer": "",
    "iteration": 0,
}

result_b = await tool_patterns_agent.ainvoke(state_b)

print(f"迭代次数：{result_b.get('iteration', 0)}")
print(f"思考记录：{len(result_b.get('reflection_history', []))} 条")
print(f"子任务委派：{len(result_b.get('pending_delegations', []))} 个")
print(f"最终答案：{result_b.get('final_answer', '未调用 TaskComplete')[:300]}")
print()
print("--- 思考历史 ---")
for i, ref in enumerate(result_b.get('reflection_history', []), 1):
    print(f"  [{i}] {ref[:150]}...")

## 4. 工具调用详情分析

看看 LLM 在完整运行过程中实际调用了哪些工具

In [ ]:
def analyze_tool_calls(result):
    """分析一次运行中所有工具调用的分布"""
    messages = result.get("messages", [])
    tool_stats = {}  # name -> count
    
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                name = tc["name"]
                tool_stats[name] = tool_stats.get(name, 0) + 1
        if hasattr(msg, "name") and msg.name in tools_by_name:
            pass  # 这是 ToolMessage，不是 AIMessage
    
    return tool_stats

print("场景 A 工具调用分布：")
stats_a = analyze_tool_calls(result_a)
for name, count in stats_a.items():
    mode = "函数式" if name in ["search_web", "fetch_page"] else (
        "Schema式" if name in ["DelegateResearch", "TaskComplete"] else "伪工具"
    )
    print(f"  {name:25s} x{count:2d}  [{mode}]")

print()
print("场景 B 工具调用分布：")
stats_b = analyze_tool_calls(result_b)
for name, count in stats_b.items():
    mode = "函数式" if name in ["search_web", "fetch_page"] else (
        "Schema式" if name in ["DelegateResearch", "TaskComplete"] else "伪工具"
    )
    print(f"  {name:25s} x{count:2d}  [{mode}]")

## 5. 四种模式对比总结

| 特性 | 模式一：函数式 | 模式二：Schema 式 | 模式三：伪工具 | 模式四：混合 |
|------|---------------|------------------|--------------|------------|
| 定义方式 | `@tool def fn()` | `@tool class X(BaseModel)` | `@tool class X(BaseModel)` | 以上任意组合 |
| 执行方式 | 在 tool_node 直接调用 | 路由到子图 / 标记状态 | 只写 state，无副作用 | 按 name 分流 |
| 外部调用 | 有 | 由子图处理 | 无 | 混合 |
| 返回值 | 函数返回值 | 子图结果 / 状态标记 | 确认消息 | 混合 |
| 典型场景 | 搜索、计算、API | 委派子任务、完成标记 | 思考、检查、决策记录 | 真实项目 |
| 项目示例 | `tavily_search` | `ConductResearch` | `think_tool` | supervisor.py |